In [0]:
import pandas as pd

path = "/Volumes/workspace/default/air_quality_data/city_day.csv"

bronze = pd.read_csv(path)

bronze = bronze.rename(columns={"PM2.5": "PM2_5"})

print("Bronze rows:", len(bronze))

Bronze rows: 29531


In [0]:
# # Clean City, Date, Year, Month and Season
silver = bronze.copy()

silver["City"] = silver["City"].str.strip()

silver["Date"] = pd.to_datetime(silver["Date"])

silver["Year"] = silver["Date"].dt.year

silver["Month"] = silver["Date"].dt.month

silver["Season"] = silver["Month"].map(
    lambda x:
        "Winter" if x in [12, 1, 2]
        else "Summer" if x in [3, 4, 5]
        else "Monsoon" if x in [6, 7, 8, 9]
        else "Post-Monsoon"
)

print("Rows:", len(silver))

display(
    silver[["City", "Date", "Year", "Month", "Season", "AQI"]].head()
)

Rows: 29531


City,Date,Year,Month,Season,AQI
Ahmedabad,2015-01-01T00:00:00.000Z,2015,1,Winter,null
Ahmedabad,2015-01-02T00:00:00.000Z,2015,1,Winter,null
Ahmedabad,2015-01-03T00:00:00.000Z,2015,1,Winter,null
Ahmedabad,2015-01-04T00:00:00.000Z,2015,1,Winter,null
Ahmedabad,2015-01-05T00:00:00.000Z,2015,1,Winter,null


In [0]:
# Remove completely empty rows + Xylene
pollutants = [
    "PM2_5", "PM10", "NO", "NO2", "NOx", "NH3",
    "CO", "SO2", "O3", "Benzene", "Toluene", "Xylene"
]

before = len(silver)

silver = silver.dropna(
    subset=pollutants + ["AQI"],
    how="all"
)

silver = silver.drop(columns=["Xylene"])

after = len(silver)

print("Deleting rows:", before - after)
print("Remaining rows:", after)

Deleting rows: 1374
Remaining rows: 28157


In [0]:
# Check the cleaning
print("Rows:", len(silver))

print("Xylene present:", "Xylene" in silver.columns)

print("Columns:", len(silver.columns))


Rows: 28157
Xylene present: False
Columns: 18


In [0]:
# Create AQI Bucket
bins = [-float("inf"), 50, 100, 200, 300, 400, float("inf")]

labels = [
    "Good",
    "Satisfactory",
    "Moderate",
    "Poor",
    "Very Poor",
    "Severe"
]

silver["AQI_Bucket"] = pd.cut(
    silver["AQI"],
    bins=bins,
    labels=labels
)

display(
    silver["AQI_Bucket"].value_counts(dropna=False)
)

AQI_Bucket
Moderate        8829
Satisfactory    8224
NaN             3307
Poor            2781
Very Poor       2337
Good            1341
Severe          1338
Name: count, dtype: int64

In [0]:
# Flag AQI above 500
silver["AQI_Above_500"] = silver["AQI"] > 500

print("Rows:", len(silver))

print(
    "Rows above 500:",
    silver["AQI_Above_500"].sum()
)

Rows: 28157
Rows above 500: 543


In [0]:
output_path = "/Volumes/workspace/default/air_quality_data/aqi_silver_python.csv"

silver.to_csv(output_path, index=False)

print("Silver data saved successfully!")
print("Rows:", len(silver))

Silver data saved successfully!
Rows: 28157
